In [4]:
# ==========================================
# DEMAND PLANNING ANALYTICS PROJECT
# ==========================================

import pandas as pd
import numpy as np

# ------------------------------------------
# LOAD DATA
# ------------------------------------------

df = pd.read_csv("demand_planning.csv")

print("Dataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())

# ------------------------------------------
# MONTH ORDER
# ------------------------------------------

month_order = [
    "Jan","Feb","Mar","Apr","May","Jun",
    "Jul","Aug","Sep","Oct","Nov","Dec"
]

df["demand_month"] = pd.Categorical(
    df["demand_month"],
    categories=month_order,
    ordered=True
)

# ------------------------------------------
# FORECAST ERROR
# ------------------------------------------

df["forecast_error"] = (
    df["actual_demand"]
    - df["forecast_demand"]
)

# ------------------------------------------
# ABSOLUTE ERROR
# ------------------------------------------

df["absolute_error"] = (
    df["forecast_error"]
    .abs()
)

# ------------------------------------------
# FORECAST ACCURACY %
# ------------------------------------------

df["forecast_accuracy"] = (
    1 -
    (
        df["absolute_error"]
        / df["actual_demand"]
    )
) * 100

df["forecast_accuracy"] = (
    df["forecast_accuracy"]
    .clip(lower=0)
)

# ------------------------------------------
# MAPE %
# ------------------------------------------

df["APE"] = (
    df["absolute_error"]
    / df["actual_demand"]
) * 100

mape = df["APE"].mean()

print("\nMAPE %:")
print(round(mape, 2))

# ------------------------------------------
# OVERALL FORECAST ACCURACY
# ------------------------------------------

overall_accuracy = (
    df["forecast_accuracy"]
    .mean()
)

print("\nOverall Forecast Accuracy %:")
print(round(overall_accuracy, 2))

# ------------------------------------------
# MONTHLY DEMAND TREND
# ------------------------------------------

monthly_demand = (
    df.groupby("demand_month", observed=True)
      .agg(
          actual_demand=("actual_demand", "sum"),
          forecast_demand=("forecast_demand", "sum")
      )
      .reset_index()
)

print("\nMonthly Demand Trend:")
print(monthly_demand)

# ------------------------------------------
# CATEGORY ANALYSIS
# ------------------------------------------

category_analysis = (
    df.groupby("category")
      .agg(
          total_actual=("actual_demand", "sum"),
          total_forecast=("forecast_demand", "sum"),
          avg_accuracy=("forecast_accuracy", "mean")
      )
      .reset_index()
)

category_analysis = category_analysis.sort_values(
    "avg_accuracy",
    ascending=False
)

print("\nCategory Analysis:")
print(category_analysis)

# ------------------------------------------
# SKU ANALYSIS
# ------------------------------------------

sku_analysis = (
    df.groupby("sku")
      .agg(
          total_actual=("actual_demand", "sum"),
          total_forecast=("forecast_demand", "sum"),
          avg_accuracy=("forecast_accuracy", "mean")
      )
      .reset_index()
)

sku_analysis = sku_analysis.sort_values(
    "avg_accuracy",
    ascending=False
)

print("\nTop SKU Analysis:")
print(sku_analysis.head(10))

# ------------------------------------------
# BEST FORECASTED SKUS
# ------------------------------------------

best_skus = (
    sku_analysis
    .sort_values(
        "avg_accuracy",
        ascending=False
    )
    .head(10)
)

print("\nBest Forecasted SKUs:")
print(best_skus)

# ------------------------------------------
# WORST FORECASTED SKUS
# ------------------------------------------

worst_skus = (
    sku_analysis
    .sort_values(
        "avg_accuracy",
        ascending=True
    )
    .head(10)
)

print("\nWorst Forecasted SKUs:")
print(worst_skus)

# ------------------------------------------
# DEMAND VARIANCE %
# ------------------------------------------

df["variance_pct"] = (
    (
        df["actual_demand"]
        - df["forecast_demand"]
    )
    / df["forecast_demand"]
) * 100

variance_summary = (
    df.groupby("category")
      .agg(
          avg_variance_pct=("variance_pct", "mean")
      )
      .reset_index()
)

print("\nDemand Variance by Category:")
print(variance_summary)

# ------------------------------------------
# TOP DEMAND SKUS
# ------------------------------------------

top_demand_skus = (
    df.groupby("sku")["actual_demand"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

print("\nTop Demand SKUs:")
print(top_demand_skus.head(10))

# ------------------------------------------
# EXPORT FILES FOR POWER BI
# ------------------------------------------

monthly_demand.to_csv(
    "monthly_demand_trend.csv",
    index=False
)

category_analysis.to_csv(
    "category_forecast_analysis.csv",
    index=False
)

sku_analysis.to_csv(
    "sku_forecast_analysis.csv",
    index=False
)

variance_summary.to_csv(
    "variance_summary.csv",
    index=False
)

top_demand_skus.to_csv(
    "top_demand_skus.csv",
    index=False
)

print("\n===================================")
print("Demand Planning Analysis Complete!")
print("Files exported successfully.")
print("===================================")

Dataset Shape:
(1200, 5)

Columns:
['sku', 'category', 'demand_month', 'actual_demand', 'forecast_demand']

First 5 Rows:
      sku     category demand_month  actual_demand  forecast_demand
0  SKU001  Fast Moving   2024-01-01           1751             1741
1  SKU001  Fast Moving          Feb           2009             1891
2  SKU001  Fast Moving          Mar           1928             1819
3  SKU001  Fast Moving          Apr           2131             1902
4  SKU001  Fast Moving          May           2387             2098

MAPE %:
8.76

Overall Forecast Accuracy %:
91.24

Monthly Demand Trend:
   demand_month  actual_demand  forecast_demand
0           Jan          76451            76493
1           Feb          81696            80724
2           Mar          87051            84536
3           Apr          88784            88905
4           May          93171            91994
5           Jun          95514            94182
6           Jul          98977            97534
7           A

/var/folders/40/ly4n03nn71n6f5x_5z8f38600000gn/T/ipykernel_41281/2037130110.py:32: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  df["demand_month"] = pd.Categorical(
